In [3]:
# 📦 Import Libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier

# 🔢 Euclidean Distance Function
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

# 🧠 KNN from Scratch
class CustomKNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train

    def predict(self, X_test):
        predictions = [self._predict(x) for x in X_test]
        return np.array(predictions)

    def _predict(self, x):
        distances = [euclidean_distance(x, x_train) for x_train in self.X_train]
        k_indices = np.argsort(distances)[:self.k]
        k_labels = [self.y_train[i] for i in k_indices]
        most_common = Counter(k_labels).most_common(1)
        return most_common[0][0]

# ✅ Evaluation Metrics from Scratch
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

def confusion_matrix_custom(y_true, y_pred, labels):
    matrix = pd.DataFrame(0, index=labels, columns=labels)
    for actual, predicted in zip(y_true, y_pred):
        matrix.loc[actual, predicted] += 1
    return matrix

def precision_recall_f1(y_true, y_pred, labels):
    cm = confusion_matrix_custom(y_true, y_pred, labels)
    precision_dict = {}
    recall_dict = {}
    f1_dict = {}

    for label in labels:
        tp = cm.loc[label, label]
        fp = cm[label].sum() - tp
        fn = cm.loc[label].sum() - tp
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        precision_dict[label] = precision
        recall_dict[label] = recall
        f1_dict[label] = f1

    return precision_dict, recall_dict, f1_dict

# 📊 Load Iris Dataset
def load_iris_dataset():
    iris = load_iris()
    X = iris.data
    y = iris.target
    labels = iris.target_names
    return X, y, labels

# 📜 Sample News Dataset (text to numeric encoding for KNN)
def load_news_dataset():
    data = {
        'text': ["government passed law", "team won match", "president met minister", "player scored goal", "new budget announced"],
        'label': ["politics", "sports", "politics", "sports", "politics"]
    }
    df = pd.DataFrame(data)
    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df['label'])

    # Simple BoW encoding
    vocab = list(set(" ".join(df['text']).split()))
    features = []
    for sentence in df['text']:
        vector = [sentence.split().count(word) for word in vocab]
        features.append(vector)
    X = np.array(features)
    y = np.array(df['label_encoded'])
    return X, y, le.classes_

# 🔍 Test & Compare
def evaluate_knn_model(X, y, labels, dataset_name="Dataset"):
    best_acc = 0
    best_k = 0
    best_split = 0
    print(f"\n📘 Evaluating: {dataset_name}")
    for split in [0.2, 0.3, 0.4]:
        for k in range(1, min(11, len(X))):
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=split, random_state=42)
            model = CustomKNN(k=k)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            acc = accuracy(y_test, y_pred)
            if acc > best_acc:
                best_acc = acc
                best_k = k
                best_split = split

    # Final model with best k & split
    print(f"\n✅ Best k = {best_k}, Split = {best_split}, Accuracy = {best_acc:.4f}")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=best_split, random_state=42)
    model = CustomKNN(k=best_k)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Custom Metrics
    acc = accuracy(y_test, y_pred)
    cm = confusion_matrix_custom(y_test, y_pred, labels=np.unique(y))
    precision, recall, f1 = precision_recall_f1(y_test, y_pred, labels=np.unique(y))

    print("\n📌 Custom Evaluation Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix:\n", cm)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    # sklearn comparison
    print("\n🤖 Comparing with scikit-learn KNN")
    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(X_train, y_train)
    y_pred_sk = knn.predict(X_test)
    print(classification_report(y_test, y_pred_sk, target_names=[str(l) for l in labels]))

# ▶ Run on Iris
X_iris, y_iris, iris_labels = load_iris_dataset()
evaluate_knn_model(X_iris, y_iris, iris_labels, "Iris Dataset")

# ▶ Run on News
X_news, y_news, news_labels = load_news_dataset()
evaluate_knn_model(X_news, y_news, news_labels, "News Dataset")



📘 Evaluating: Iris Dataset

✅ Best k = 1, Split = 0.2, Accuracy = 1.0000

📌 Custom Evaluation Metrics:
Accuracy: 1.0000
Confusion Matrix:
     0  1   2
0  10  0   0
1   0  9   0
2   0  0  11
Precision: {np.int64(0): np.float64(0.9999999999), np.int64(1): np.float64(0.9999999998888889), np.int64(2): np.float64(0.9999999999090909)}
Recall: {np.int64(0): np.float64(0.9999999999), np.int64(1): np.float64(0.9999999998888889), np.int64(2): np.float64(0.9999999999090909)}
F1-score: {np.int64(0): np.float64(0.9999999994), np.int64(1): np.float64(0.9999999993888888), np.int64(2): np.float64(0.9999999994090909)}

🤖 Comparing with scikit-learn KNN
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00     

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
